# PGA-UNet — Test & Visualization
**Checkpoint**: `1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY`

Chạy tuần tự: **Setup → Test → Visualization**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
%cd /content
import os, gdown

if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git
else:
    !cd Prompt-Guided-XRay-Segmentation && git pull -q

!gdown 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW -O /content/dataset_BTXRD.zip -q
!unzip -oq /content/dataset_BTXRD.zip -d /content/
!rsync -a /content/dataset_BTXRD/ /content/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/ 2>/dev/null

%cd /content/Prompt-Guided-XRay-Segmentation
!pip install -q tqdm opencv-python matplotlib gdown scipy

CKPT_ID   = '1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY'
CKPT_PATH = 'checkpoints/pga_unet_expB_best.pth'
os.makedirs('checkpoints', exist_ok=True)
if not os.path.exists(CKPT_PATH):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_PATH, quiet=False)
assert os.path.exists(CKPT_PATH)
print(f'\n✅ Setup xong  |  {os.path.getsize(CKPT_PATH)//1024} KB')

In [ ]:
# ── Test: 3 prompt modes — image-level merging ───────────────────────────────
import os, cv2, csv, json as _json
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR  = 'dataset_BTXRD/test/images'
JSON_DIR = 'dataset_BTXRD/test/annotations'
os.makedirs('results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded  device={DEVICE}')

def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95), np.percentile(d2,95)))

def calc_metrics_img(prob_np, gt_np):
    pm = (prob_np > 0.5).astype(np.float32)
    gm = (gt_np  > 0.5).astype(np.float32); eps = 1e-6
    tp = (pm * gm).sum(); fp = (pm * (1-gm)).sum(); fn = ((1-pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        bbox_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2) / bbox_diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS  = ['dice', 'iou', 'precision', 'recall', 'hd95', 'cbl']
HDRS  = ['Dice↑', 'IoU↑', 'Prec↑', 'Rec↑', 'HD95↓', 'CBL↑']
MODES = ['zoom_out', 'shift', 'mixed_7_3']
all_image_records = {}   # mode → list of image-level dicts (sorted by img_name)
csv_rows = []

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)

    # Accumulate per original-image
    groups = OrderedDict()   # img_name → accumulator dict
    for i, (img_t, mask_t, prompt_t) in enumerate(tqdm(loader, desc=f'[{mode}]')):
        img_name, _ = ds.all_samples[i]
        gt_np     = mask_t[0, 0].numpy()
        prompt_np = prompt_t[0, 0].numpy()

        with torch.no_grad():
            prob = torch.sigmoid(model(img_t.to(DEVICE), prompt_t.to(DEVICE)))[0, 0].cpu().numpy()

        if img_name not in groups:
            groups[img_name] = dict(img=img_t[0, 0].numpy(),
                                    gt_union=gt_np.copy(),
                                    prob_max=prob.copy(),
                                    prompts=[prompt_np])
        else:
            g = groups[img_name]
            np.maximum(g['gt_union'],  gt_np, out=g['gt_union'])   # union GT polygons
            np.maximum(g['prob_max'],  prob,  out=g['prob_max'])    # max-merge predictions
            g['prompts'].append(prompt_np)

    # Build sorted image-level records
    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]
        m = calc_metrics_img(g['prob_max'], g['gt_union'])
        img_recs.append(dict(img_name=img_name,
                             img=g['img'],
                             gt=g['gt_union'],
                             prob=g['prob_max'],
                             prompts=g['prompts'],
                             n_samples=len(g['prompts']),
                             **m))

    all_image_records[mode] = img_recs
    m_avg  = {k: np.mean([r[k] for r in img_recs]) for k in KEYS}
    n_imgs = len(img_recs)
    n_samp = sum(r['n_samples'] for r in img_recs)
    csv_rows.append([mode] + [f'{m_avg[k]:.4f}' for k in KEYS] + [str(n_imgs), str(n_samp)])

# Bảng kết quả
bar = '=' * 82
print(f'\n{bar}\n  PGA-UNet — Image-level metrics (GT union + max-merge predictions)\n{bar}')
print(f"  {'Mode':<16}" + ''.join(f'{h:>8}' for h in HDRS) + f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in csv_rows:
    print(f"  {row[0]:<16}" + ''.join(f'{row[i+1]:>8}' for i in range(len(KEYS))) + f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open('results/pga_extended_test_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['mode'] + KEYS + ['N_img', 'N_samples'])
    w.writerows(csv_rows)
print('\n✅ CSV: results/pga_extended_test_results.csv')


In [ ]:
# ── Visualization: ≥10 ảnh có từ 2 GT polygon trở lên (zoom_out) ─────────────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Chạy cell Test trước'

N_MIN  = 10   # tối thiểu bao nhiêu ảnh cần show
MIN_GT = 2    # số polygon tối thiểu

recs = [r for r in all_image_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs = recs[:max(N_MIN, len(recs))]   # show tất cả, ít nhất N_MIN
N_SHOW = len(recs)

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'PGA-UNet — {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np  = rec['img'] * 0.5 + 0.5
    gt_np   = (rec['gt']   > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum()
    fp = (pred_np * (1 - gt_np)).sum()
    fn = ((1 - pred_np) * gt_np).sum()
    e  = 1e-6
    dice = float((2*tp+e) / (2*tp+fp+fn+e))
    iou  = float((tp+e)   / (tp+fp+fn+e))
    pre  = float((tp+e)   / (tp+fp+e))
    rec_ = float((tp+e)   / (tp+fn+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    # Col 0: ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1: ảnh + tất cả prompt (max-merged)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2: merged prediction overlay (đỏ)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3: GT union overlay (xanh lá)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4: TP/FP/FN
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np       * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np)   * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np   * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)
